# DengueRadar — 01 | Limpeza e integração de dados

**Projeto FIAP | CitizenTech | Ciência de Dados aplicada à saúde pública**

## Contexto e problema
A dengue exige monitoramento contínuo porque aumentos de casos podem pressionar a rede de saúde. Famílias, profissionais de saúde e gestores públicos são afetados. A análise de séries históricas pode ajudar a identificar padrões sazonais e oferecer subsídios ao planejamento preventivo.

**Pergunta do projeto:** até que ponto a semana epidemiológica e as condições climáticas ajudam a estimar o número agregado de casos de dengue?

A aplicação de IA aqui é **exploratória**: ela não identifica focos reais, não confirma surtos nem substitui a vigilância epidemiológica. Resultados dependem da qualidade e do período dos dados.

## Origem do dataset e dicionário de variáveis
- **SINAN/DATASUS (exportação TABNET):** contagens notificadas de casos por ano e semana, conforme filtros da consulta realizada. A definição do indicador e a abrangência geográfica dependem da exportação efetiva.
- **INMET:** registros de temperatura e precipitação de estações meteorológicas. É necessário documentar a estação, cobertura e resolução temporal usadas.
- **Alvo (`casos`):** número de casos agregados na semana para o recorte selecionado, obtido no TABNET.
- **Preditor (`semana_epidem`):** posição da semana no ano (representa a sazonalidade).
- **Preditor (`temp_media`):** temperatura média agregada para a semana (°C).
- **Preditor (`precipitacao_total`):** precipitação somada para a semana (mm).
- **Coluna de organização (`ano`):** permite ordenar as observações cronologicamente; não é feature do modelo atual.

> Limitação essencial: quando a temperatura e a precipitação do próprio período são usadas como entradas, o modelo caracteriza uma **estimativa condicionada a valores climáticos conhecidos ou simulados**, não uma previsão operacional futura sem previsão meteorológica. Também não tratamos uma ausência no TABNET como zero automaticamente.

**Como reproduzir:** coloque `data/raw/datasus_dengue.csv` e `data/raw/inmet_clima.csv` na raiz do projeto; execute este notebook antes do notebook 02.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path.cwd().parent if Path.cwd().name.lower() in ('notebooks', 'notebook') else Path.cwd()
RAW = BASE / 'data' / 'raw'
PROCESSED = BASE / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)
assert (RAW / 'datasus_dengue.csv').exists(), f"Arquivo ausente: {RAW / 'datasus_dengue.csv'}"
assert (RAW / 'inmet_clima.csv').exists(), f"Arquivo ausente: {RAW / 'inmet_clima.csv'}"
print('Diretório do projeto:', BASE)

Diretório do projeto: c:\Users\mathe\Documents\Fiap\2026\1 Semestre\Fase 7\GS\dengueradar


## 1. Leitura e pré-processamento dos casos (SINAN/DATASUS)
Lemos o CSV no formato da exportação TABNET, convertemos a tabela larga (anos nas colunas) para registros por ano/semana, validamos contagens e mantemos valores não reconhecidos como ausentes. Não removemos linhas de rodapé por quantidade fixa: registros que não identificam uma semana válida são descartados.

In [ ]:
# A quantidade de linhas de rodapé varia por exportação TABNET: não usar skipfooter fixo.
df_dengue = pd.read_csv(RAW / 'datasus_dengue.csv', sep=';', encoding='latin1', engine='python', dtype=str)
df_dengue.columns = df_dengue.columns.str.strip()
coluna_semana = df_dengue.columns[0]
colunas_anos = [c for c in df_dengue.columns if str(c).strip().isdigit() and len(str(c).strip()) == 4]
if not colunas_anos:
    raise ValueError('Não encontrei colunas de anos (ex.: 2023, 2024) no CSV DATASUS. Confira a exportação.')

df_casos = df_dengue.melt(id_vars=[coluna_semana], value_vars=colunas_anos, var_name='ano', value_name='casos_raw')
df_casos['semana_epidem'] = pd.to_numeric(df_casos[coluna_semana].astype(str).str.extract(r'(?i)semana\s*0*(\d{1,2})')[0], errors='coerce')
df_casos['ano'] = pd.to_numeric(df_casos['ano'], errors='coerce')
# Nota: '-' pode indicar zero, ausência ou dado não disponível, conforme o relatório de origem.
# Preservamos como NaN até confirmar essa codificação nos metadados.
valor = df_casos['casos_raw'].astype(str).str.strip().str.replace('.', '', regex=False)
valor = valor.str.replace(',', '.', regex=False)
df_casos['casos'] = pd.to_numeric(valor, errors='coerce')
df_casos = df_casos.dropna(subset=['ano', 'semana_epidem'])
df_casos = df_casos[df_casos['semana_epidem'].between(1, 53)]
df_casos['ano'] = df_casos['ano'].astype(int)
df_casos['semana_epidem'] = df_casos['semana_epidem'].astype(int)
df_casos = df_casos[['ano', 'semana_epidem', 'casos']]
if df_casos.duplicated(['ano', 'semana_epidem']).any():
    raise ValueError('Existem ano/semana duplicados no DATASUS. Verifique se a tabela contém mais de um território ou categoria.')
if (df_casos['casos'].dropna() < 0).any():
    raise ValueError('Foram encontrados casos negativos.')
print(f'Linhas válidas de período: {len(df_casos)} | casos ausentes: {df_casos.casos.isna().sum()}')
display(df_casos.head())

Linhas válidas de período: 159 | casos ausentes: 104


,ano,semana_epidem,casos
0,2024,1,179.0
1,2024,2,NaN
2,2024,3,NaN
3,2024,4,NaN
4,2024,5,NaN


## 2. Leitura e pré-processamento climático (INMET)
Convertemos datas e medidas numéricas, calculamos agregações semanais e avaliamos a cobertura das observações.

**Atenção à semana:** o agrupamento implementado usa semanas de domingo a sábado com numeração ancorada em 1º de janeiro. Isso **não garante correspondência automática** com a convenção do TABNET; precisamos verificar especialmente a virada de ano.

**Atenção à estação:** o limite de 118 leituras pressupõe dados **horários de uma estação** (70% de 168 registros). Para dados diários ou múltiplas estações, altere a regra e documente a decisão. É necessário assegurar que a variável de precipitação represente um acumulado de intervalo e não um acumulado repetido em cada leitura.

In [ ]:
df_clima = pd.read_csv(RAW / 'inmet_clima.csv', sep=';', encoding='latin1', skiprows=8, decimal=',', low_memory=False)
df_clima.columns = df_clima.columns.str.strip()
col_data = next((c for c in df_clima.columns if c.lower().startswith('data')), None)
col_temp = next((c for c in df_clima.columns if 'TEMPERATURA DO AR - BULBO SECO' in c.upper()), None)
col_chuva = next((c for c in df_clima.columns if 'PRECIPITAÇÃO TOTAL' in c.upper()), None)
if not all([col_data, col_temp, col_chuva]):
    raise ValueError(f'Colunas INMET não reconhecidas: {df_clima.columns.tolist()}')
df_clima['Data'] = pd.to_datetime(df_clima[col_data], errors='coerce', dayfirst=False)
for origem, destino in [(col_temp, 'temp_media'), (col_chuva, 'precipitacao_total')]:
    df_clima[destino] = pd.to_numeric(df_clima[origem].astype(str).str.replace(',', '.', regex=False), errors='coerce')
df_clima = df_clima.dropna(subset=['Data']).copy()

def semana_domingo(data):
    ano = data.year
    primeiro = pd.Timestamp(year=ano, month=1, day=1)
    inicio = primeiro - pd.Timedelta(days=(primeiro.weekday() + 1) % 7)
    return pd.Series({'ano': ano, 'semana_epidem': 1 + (data - inicio).days // 7})

semanas = df_clima['Data'].apply(semana_domingo)
df_clima[['ano','semana_epidem']] = semanas[['ano','semana_epidem']]
# ATENÇÃO: soma horária da chuva só é válida para série de uma estação e cobertura adequada.
# Dados horários de múltiplas estações exigem agregação espacial prévia.
clima_semanal = df_clima.groupby(['ano','semana_epidem'], as_index=False).agg(
    temp_media=('temp_media','mean'),
    precipitacao_total=('precipitacao_total', lambda s: s.sum(min_count=1)),
    medicoes_temp=('temp_media','count'),
    medicoes_chuva=('precipitacao_total','count')
)
# Padrão mínimo ilustrativo: pelo menos 70% das 168 leituras horárias esperadas por semana.
# Ajustar em caso de dados diários ou múltiplas estações e documentar essa escolha.
MIN_LEITURAS = 118
clima_semanal.loc[clima_semanal.medicoes_temp < MIN_LEITURAS, 'temp_media'] = np.nan
clima_semanal.loc[clima_semanal.medicoes_chuva < MIN_LEITURAS, 'precipitacao_total'] = np.nan
print('Semanas clima:', len(clima_semanal), '| Semanas com clima incompleto:', clima_semanal[['temp_media','precipitacao_total']].isna().any(axis=1).sum())
display(clima_semanal.head())

Semanas clima: 53 | Semanas com clima incompleto: 2


,ano,semana_epidem,temp_media,precipitacao_total,medicoes_temp,medicoes_chuva
0,2025,1,NaN,NaN,96,96
1,2025,2,22.126786,13.4,168,168
2,2025,3,22.317262,36.0,168,168
3,2025,4,25.947904,144.2,167,167
4,2025,5,22.354167,183.4,168,168


## 3. União temporal, remoção justificada de ausências e exportação
Unimos as séries por ano e semana, descartamos registros incompletos para as três variáveis exigidas no treinamento e ordenamos cronologicamente. O número removido é relatado, pois perdas seletivas de observações podem introduzir viés.

In [ ]:
df_final = df_casos.merge(clima_semanal[['ano','semana_epidem','temp_media','precipitacao_total']], on=['ano','semana_epidem'], how='inner', validate='one_to_one')
antes = len(df_final)
df_final = df_final.dropna(subset=['casos','temp_media','precipitacao_total'])
df_final = df_final.sort_values(['ano','semana_epidem']).reset_index(drop=True)
if df_final.empty:
    raise ValueError('Base vazia após união e limpeza. Verifique anos, semanas e cobertura da estação meteorológica.')
if df_final.duplicated(['ano','semana_epidem']).any():
    raise ValueError('Ano/semana não é único.')
saida = PROCESSED / 'dataset_dengueradar_limpo.csv'
df_final.to_csv(saida, index=False)
print(f'Exportado: {saida} | {len(df_final)} semanas | {antes-len(df_final)} semanas removidas por dados ausentes')
print('Período:', (int(df_final.ano.min()), int(df_final.semana_epidem.min())), 'até', (int(df_final.ano.max()), int(df_final.semana_epidem.max())))
display(df_final.tail())

Exportado: c:\Users\mathe\Documents\Fiap\2026\1 Semestre\Fase 7\GS\dengueradar\data\processed\dataset_dengueradar_limpo.csv | 51 semanas | 2 semanas removidas por dados ausentes
Período: (2025, 2) até (2025, 52)


,ano,semana_epidem,casos,temp_media,precipitacao_total
46,2025,48,347.0,20.274096,9.0
47,2025,49,305.0,22.601307,7.4
48,2025,50,263.0,23.039521,128.4
49,2025,51,286.0,21.548810,14.4
50,2025,52,257.0,27.086145,22.8


## 4. Auditoria simples do dataset tratado
A tabela abaixo resume o volume, a cobertura temporal, as ausências remanescentes e eventuais saltos na sequência semanal. Essa auditoria auxilia a decisão sobre a viabilidade da modelagem.

In [ ]:
print('Amostras após ETL:', len(df_final))
print('Anos disponíveis:', sorted(df_final['ano'].unique().tolist()))
print('Semanas por ano:')
display(df_final.groupby('ano').agg(semanas=('semana_epidem','count'), casos_totais=('casos','sum')).reset_index())
print('Nulos no conjunto tratado:')
display(df_final.isna().sum().rename('quantidade'))
print('Resumo descritivo:')
display(df_final[['casos','temp_media','precipitacao_total']].describe().round(2))
periodos = df_final[['ano','semana_epidem']].drop_duplicates().sort_values(['ano','semana_epidem'])
print('Observação: confirme se todas as semanas que deveriam existir foram preservadas (ausências podem não ser aleatórias).')

Amostras após ETL: 51
Anos disponíveis: [2025]
Semanas por ano:


,ano,semanas,casos_totais
0,2025,51,71575.0


Nulos no conjunto tratado:


ano                   0
semana_epidem         0
casos                 0
temp_media            0
precipitacao_total    0
Name: quantidade, dtype: int64

Resumo descritivo:


,casos,temp_media,precipitacao_total
count,51.00,51.00,51.00
mean,1403.43,20.33,27.46
std,1572.86,3.08,38.65
min,257.00,14.42,0.00
25%,315.50,18.10,1.40
50%,434.00,20.16,14.40
75%,2093.50,22.34,33.50
max,5145.00,27.09,183.40


Observação: confirme se todas as semanas que deveriam existir foram preservadas (ausências podem não ser aleatórias).


## 5. Limitações e decisões de limpeza
- Não assumimos que `-`, células vazias ou valores não interpretáveis signifiquem zero casos.
- A junção exige uma observação por ano/semana em cada fonte e remove períodos sem clima ou casos, o que pode afetar a representatividade.
- DATASUS/SINAN pode sofrer atrasos e revisões de notificação; comparações próximas à data atual exigem cautela.
- Sem validar estação, região e calendário epidemiológico não podemos afirmar correspondência espacial/temporal perfeita.
- O arquivo final é salvo em `data/processed/dataset_dengueradar_limpo.csv`; ele será a única base de treinamento utilizada no notebook 02.

**Próxima etapa:** dois modelos supervisionados, divisões temporais de treino/validação/teste, métricas comparativas e análise crítica.